<a href="https://colab.research.google.com/github/PaloMoshoeshoe/Stroke-prediction/blob/main/stroke_model_comparison_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stroke model comparison and surrogate fidelity

This notebook implements the supplied Chapter 3 design on the Kaggle Stroke Prediction Dataset. It compares four intrinsically interpretable model families (L1 logistic regression, depth-limited CART, EBM when available, and GOSDT when available) with three black-box classifiers, then measures global surrogate fidelity, including how often a surrogate repeats a black-box model's errors.

**Data limitation:** the supplied methodology identifies serious provenance concerns for this public dataset. Treat results only as a methodological benchmark; they are not clinical evidence or suitable for clinical decisions.

Run all cells in order. The full design is 10-fold stratified CV repeated five times (50 splits). It can take several hours on CPU, especially when GOSDT is available: a full run fits up to seven base models plus 12 surrogates on every split (up to 950 fits, including up to 200 GOSDT fits). Set `QUICK_RUN = True` for a one-repeat dry run, then set it to `False` for the full analysis.

## 1. Install core dependencies and try optional interpretable models

InterpretML (EBM) and GOSDT are installed separately. Their availability depends on the Colab Python/runtime; if either installation or import fails, the notebook reports the omission and continues with the remaining models.

In [7]:
%pip -q install "numpy>=1.24" "pandas>=2.0" "scipy>=1.10" "scikit-learn>=1.7"

import subprocess, sys
for optional_package in ("interpret", "gosdt"):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", optional_package], check=True)
        print(f"Installed optional package: {optional_package}")
    except Exception as exc:
        print(f"Optional package {optional_package} not installed: {exc}")

Installed optional package: interpret
Optional package gosdt not installed: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'gosdt']' returned non-zero exit status 1.


## 2. Choose CSV upload or Google Drive path

Upload the dataset file named `Stroke.csv`, or mount Drive and enter its path. This cell does not download data or require Kaggle credentials.

In [8]:
from pathlib import Path
import io
import pandas as pd

DATA_SOURCE = input("Type upload to upload a CSV, or drive to use a Google Drive path [upload/drive]: ").strip().lower()
if DATA_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No CSV file was uploaded.")

    # Find the uploaded file key (handling automatic renaming like 'Stroke (4).csv')
    uploaded_filename = list(uploaded.keys())[0]
    if not ("stroke" in uploaded_filename.lower() and uploaded_filename.endswith(".csv")):
        raise ValueError(f"Please upload the dataset with 'Stroke.csv' in its name. Received: {uploaded_filename}")

    raw_df = pd.read_csv(io.BytesIO(uploaded[uploaded_filename]))
elif DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    CSV_PATH = input("Enter the full path to Stroke.csv in Drive: ").strip()
    if not Path(CSV_PATH).is_file():
        raise FileNotFoundError(f"CSV not found: {CSV_PATH}")
    raw_df = pd.read_csv(CSV_PATH)
else:
    raise ValueError("Choose exactly 'upload' or 'drive'.")

print(f"Loaded {len(raw_df):,} rows and {raw_df.shape[1]} columns")
display(raw_df.head())

Type upload to upload a CSV, or drive to use a Google Drive path [upload/drive]: upload


Saving Stroke.csv to Stroke (6).csv
Loaded 5,110 rows and 12 columns


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 3. Validate and preprocess the raw data

The missing-BMI flag is created from each row's missingness. Imputation, scaling, and encoding are fitted later inside each training fold to prevent leakage.

In [9]:
import numpy as np
from core import prepare_data

X, y = prepare_data(raw_df)
print(f"Rows: {len(y):,}; raw features: {X.shape[1]}; stroke prevalence: {y.mean():.3%}")
print(f"Missing BMI values: {int(X['bmi'].isna().sum())}")
print("Target counts:", dict(zip(*np.unique(y, return_counts=True))))

Rows: 5,110; raw features: 11; stroke prevalence: 4.873%
Missing BMI values: 201
Target counts: {np.int64(0): np.int64(4861), np.int64(1): np.int64(249)}


## 4. Define preprocessing and model families

Preprocessing is fit fold by fold. Base-level model hyperparameters follow the supplied methodology; no hyperparameter search is performed. EBM and GOSDT are included only when available. GOSDT learns thresholds on each training fold, uses a small gradient-boosting reference model, and has a depth limit and time limit.

In [10]:
%%writefile core.py
"""Core CV, model, fidelity, and corrected interval implementation."""
import warnings
import numpy as np
import pandas as pd
from scipy.stats import t
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             recall_score, accuracy_score)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

SEED = 42
NUMERIC = ["age", "avg_glucose_level", "bmi"]
BINARY = ["hypertension", "heart_disease", "bmi_was_missing"]
CATEGORICAL = ["gender", "ever_married", "work_type", "residence_type", "smoking_status"]
BASE_INTERPRETABLE = ["L1 logistic regression", "Depth-limited CART"]
BASE_BLACK_BOX = ["Random forest", "Gradient boosting", "Multilayer perceptron"]


def prepare_data(df):
    """Validate the Kaggle schema and return raw feature frame and binary target."""
    data = df.copy()
    data.columns = [str(c).strip().lower() for c in data.columns]
    data = data.drop(columns=["id"], errors="ignore")
    required = set(NUMERIC + ["hypertension", "heart_disease"] + CATEGORICAL + ["stroke"])
    missing = sorted(required - set(data.columns))
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")
    ordered = NUMERIC + ["hypertension", "heart_disease"] + CATEGORICAL + ["stroke"]
    data = data[ordered].copy()
    target = pd.to_numeric(data.pop("stroke"), errors="raise")
    if target.isna().any() or not set(target.unique()).issubset({0, 1}):
        raise ValueError("stroke must contain only non-missing 0 and 1 values")
    data["bmi_was_missing"] = data["bmi"].isna().astype("int8")
    return data, target.astype(int).to_numpy()


def build_preprocessor():
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:  # scikit-learn <1.2
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer([
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                               ("scale", StandardScaler())]), NUMERIC),
        ("binary", "passthrough", BINARY),
        ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                  ("onehot", encoder)]), CATEGORICAL),
    ], remainder="drop")


def available_models():
    names = BASE_INTERPRETABLE + BASE_BLACK_BOX
    try:
        from interpret.glassbox import ExplainableBoostingClassifier  # noqa: F401
        names.insert(1, "Explainable Boosting Machine")
    except Exception as exc:
        warnings.warn(f"EBM unavailable; omitting it: {exc}")
    try:
        from gosdt import ThresholdGuessBinarizer, GOSDTClassifier  # noqa: F401
        names.insert(names.index("Random forest"), "GOSDT")
    except Exception as exc:
        warnings.warn(f"GOSDT unavailable; omitting it: {exc}")
    return names


def build_models(seed=SEED):
    models = {
        "L1 logistic regression": LogisticRegression(penalty="l1", solver="liblinear",
            class_weight="balanced", max_iter=3000, random_state=seed),
        "Depth-limited CART": DecisionTreeClassifier(max_depth=4,
            class_weight="balanced", random_state=seed),
        "Random forest": RandomForestClassifier(n_estimators=300,
            class_weight="balanced_subsample", min_samples_leaf=2, n_jobs=-1, random_state=seed),
        "Gradient boosting": GradientBoostingClassifier(n_estimators=100,
            learning_rate=0.05, max_depth=2, random_state=seed),
        "Multilayer perceptron": MLPClassifier(hidden_layer_sizes=(64, 32), alpha=0.001,
            max_iter=500, early_stopping=True, random_state=seed),
    }
    try:
        from interpret.glassbox import ExplainableBoostingClassifier
        models["Explainable Boosting Machine"] = ExplainableBoostingClassifier(
            interactions=2, outer_bags=8, max_rounds=5000, n_jobs=1, random_state=seed)
    except Exception:
        pass
    try:
        from gosdt import ThresholdGuessBinarizer, GOSDTClassifier  # noqa: F401
        models["GOSDT"] = "gosdt"
    except Exception:
        pass
    return models


def _probability(model, X):
    if hasattr(model, "predict_proba"):
        classes = list(model.classes_)
        return model.predict_proba(X)[:, classes.index(1)]
    scores = model.decision_function(X)
    return scores if np.ndim(scores) == 1 else scores[:, list(model.classes_).index(1)]


def _fit_predict(name, estimator, X_train, y_train, X_test, seed):
    weights = compute_sample_weight(class_weight="balanced", y=y_train)
    if name == "GOSDT":
        from gosdt import ThresholdGuessBinarizer, GOSDTClassifier
        # GOSDT expects Boolean inputs. Learn candidate thresholds from the
        # training fold only, then apply those same thresholds to validation.
        xtr = pd.DataFrame(X_train).reset_index(drop=True)
        xte = pd.DataFrame(X_test, columns=xtr.columns).reset_index(drop=True)
        thresholder = ThresholdGuessBinarizer(n_estimators=40, max_depth=1,
                                               random_state=seed)
        thresholder.set_output(transform="pandas")
        xtr_bin = thresholder.fit_transform(xtr, pd.Series(y_train))
        xte_bin = thresholder.transform(xte)

        # A small reference ensemble supplies warm-start labels for the search.
        warm = GradientBoostingClassifier(n_estimators=40, max_depth=1,
                                          random_state=seed)
        warm.fit(xtr_bin, y_train, sample_weight=weights)
        model = GOSDTClassifier(regularization=0.03, balance=True,
                                depth_budget=4, time_limit=30, verbose=False)
        model.fit(xtr_bin, y_train, y_ref=warm.predict(xtr_bin))
        pred = np.asarray(model.predict(xte_bin), dtype=int).ravel()
        return model, thresholder, pred, pred.astype(float)

    model = clone(estimator)
    if name in ("Explainable Boosting Machine", "Gradient boosting", "Multilayer perceptron"):
        model.fit(X_train, y_train, sample_weight=weights)
    else:
        model.fit(X_train, y_train)
    pred = np.asarray(model.predict(X_test), dtype=int).ravel()
    return model, None, pred, _probability(model, X_test)


def _predict_labels(name, model, extra, X):
    if name == "GOSDT":
        X = pd.DataFrame(X)
        X_bin = extra.transform(X)
        return np.asarray(model.predict(X_bin), dtype=int).ravel()
    return np.asarray(model.predict(X), dtype=int).ravel()


def run_experiment(X, y, n_splits=10, n_repeats=5, seed=SEED):
    """Run fold-local preprocessing, performance evaluation, and surrogate fidelity."""
    y = np.asarray(y, dtype=int)
    if len(np.unique(y)) != 2:
        raise ValueError("Repeated stratified ROC-AUC evaluation requires both target classes")
    models_available = build_models(seed)
    interpretable = [n for n in models_available if n in BASE_INTERPRETABLE or
                     n in ("Explainable Boosting Machine", "GOSDT")]
    black_box = [n for n in BASE_BLACK_BOX if n in models_available]
    cv = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=seed)
    model_rows, fidelity_rows = [], []
    for split_id, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        xtr_raw, xte_raw = X.iloc[train_idx], X.iloc[test_idx]
        ytr, yte = y[train_idx], y[test_idx]
        prep = build_preprocessor()
        xtr, xte = prep.fit_transform(xtr_raw, ytr), prep.transform(xte_raw)
        fold_models = build_models(seed + split_id)
        bb_predictions = {}
        for name, estimator in fold_models.items():
            try:
                model, extra, pred, score = _fit_predict(name, estimator, xtr, ytr, xte, seed + split_id)
                model_rows.append({"split": split_id, "model": name,
                    "roc_auc": roc_auc_score(yte, score),
                    "average_precision": average_precision_score(yte, score),
                    "sensitivity": recall_score(yte, pred, zero_division=0)})
                if name in black_box:
                    bb_predictions[name] = (_predict_labels(name, model, extra, xtr), pred)
            except Exception as exc:
                if name in ("Explainable Boosting Machine", "GOSDT"):
                    warnings.warn(f"{name} failed on split {split_id} and is omitted for this split: {exc}")
                    continue
                raise
        # Only fit surrogates corresponding to interpretable estimators available in this runtime.
        for bb, (target_train, target_test) in bb_predictions.items():
            for surrogate_name in interpretable:
                try:
                    surrogate, extra, surrogate_pred, _ = _fit_predict(surrogate_name,
                        build_models(seed + split_id)[surrogate_name],
                        xtr, target_train, xte, seed + split_id + 1000)
                    wrong = target_test != yte
                    mode = int(pd.Series(target_train).mode().iloc[0])
                    baseline = np.full(len(yte), mode)
                    fidelity_rows.append({"split": split_id, "black_box": bb,
                        "surrogate": surrogate_name,
                        "agreement": accuracy_score(target_test, surrogate_pred),
                        "error_agreement": (float(np.mean(surrogate_pred[wrong] == target_test[wrong]))
                                            if wrong.any() else np.nan),
                        "majority_baseline_agreement": accuracy_score(target_test, baseline),
                        "black_box_error_count": int(wrong.sum())})
                except Exception as exc:
                    if surrogate_name in ("Explainable Boosting Machine", "GOSDT"):
                        warnings.warn(f"{surrogate_name} surrogate failed on split {split_id}: {exc}")
                        continue
                    raise
        print(f"Completed split {split_id}/{n_splits * n_repeats}")
    return pd.DataFrame(model_rows), pd.DataFrame(fidelity_rows)


def corrected_resampled_ci(values, n_splits=10, n_repeats=5, test_train_ratio=None,
                           confidence=0.95):
    """Nadeau–Bengio corrected resampled t interval for a vector of fold values."""
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if not len(vals):
        return {"mean": np.nan, "ci_low": np.nan, "ci_high": np.nan, "n": 0}
    mean = float(vals.mean())
    if len(vals) < 2:
        return {"mean": mean, "ci_low": np.nan, "ci_high": np.nan, "n": len(vals)}
    ratio = test_train_ratio if test_train_ratio is not None else 1 / (n_splits - 1)
    correction = 1 / len(vals) + ratio
    se = np.sqrt(correction * vals.var(ddof=1))
    critical = t.ppf((1 + confidence) / 2, df=len(vals) - 1)
    return {"mean": mean, "ci_low": float(mean - critical * se),
            "ci_high": float(mean + critical * se), "n": len(vals)}


def summarize_results(model_rows, fidelity_rows, n_splits=10, n_repeats=5):
    performance = model_rows.groupby("model").agg(
        roc_auc_mean=("roc_auc", "mean"), roc_auc_sd=("roc_auc", "std"),
        average_precision_mean=("average_precision", "mean"),
        sensitivity_mean=("sensitivity", "mean"), n_folds=("split", "count"))
    comparisons = []
    if not model_rows.empty:
        wide = model_rows.pivot(index="split", columns="model", values="roc_auc")
        interpretable = [n for n in ["L1 logistic regression", "Explainable Boosting Machine",
            "Depth-limited CART", "GOSDT"] if n in wide]
        black_box = [n for n in BASE_BLACK_BOX if n in wide]
        for a in interpretable:
            for b in black_box:
                ci = corrected_resampled_ci(wide[a] - wide[b], n_splits, n_repeats)
                comparisons.append({"interpretable_model": a, "black_box_model": b,
                                    "mean_difference": ci["mean"], "ci_low": ci["ci_low"],
                                    "ci_high": ci["ci_high"], "n_folds": ci["n"]})
    auc_comparisons = pd.DataFrame(comparisons)
    fidelity_summaries = []
    if not fidelity_rows.empty:
        for (bb, surrogate), group in fidelity_rows.groupby(["black_box", "surrogate"]):
            row = {"black_box": bb, "surrogate": surrogate}
            for metric in ("agreement", "error_agreement"):
                ci = corrected_resampled_ci(group[metric], n_splits, n_repeats)
                row[f"{metric}_mean"] = ci["mean"]
                row[f"{metric}_ci_low"] = ci["ci_low"]
                row[f"{metric}_ci_high"] = ci["ci_high"]
                row[f"{metric}_n_folds"] = ci["n"]
            row["majority_baseline_mean"] = group["majority_baseline_agreement"].mean()
            row["mean_black_box_errors"] = group["black_box_error_count"].mean()
            fidelity_summaries.append(row)
    return {"performance_summary": performance,
            "auc_comparisons": auc_comparisons,
            "fidelity_summary": pd.DataFrame(fidelity_summaries)}


Overwriting core.py


## 5. Repeated stratified cross-validation and model comparison

The same repeated stratified folds are used for every model. Optional EBM/GOSDT errors are warned about; core model errors stop the run so incomplete results are not mistaken for full results. GOSDT returns hard labels in this adapter, so its ROC-AUC and average precision are coarser than score-based estimates.

In [11]:
from core import build_models, available_models, run_experiment
import time

SEED = 42
QUICK_RUN = False  # True = 10 splits; False = methodology's 50 splits
N_SPLITS = 10
N_REPEATS = 1 if QUICK_RUN else 5
print("Models available in this runtime:", available_models())
print(f"Design: {N_SPLITS} folds × {N_REPEATS} repeats")

started = time.time()
model_results, fidelity_results = run_experiment(
    X, y, n_splits=N_SPLITS, n_repeats=N_REPEATS, seed=SEED)
print(f"Finished in {(time.time()-started)/60:.1f} minutes")
print(f"Performance rows: {len(model_results)}; fidelity rows: {len(fidelity_results)}")

Models available in this runtime: ['L1 logistic regression', 'Explainable Boosting Machine', 'Depth-limited CART', 'Random forest', 'Gradient boosting', 'Multilayer perceptron']
Design: 10 folds × 5 repeats


/content/core.py:71: UserWarning: GOSDT unavailable; omitting it: No module named 'gosdt'
  warnings.warn(f"GOSDT unavailable; omitting it: {exc}")
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid th

Completed split 1/50


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead o

Completed split 2/50


...

Completed split 48/50


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead o

Completed split 49/50


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead o

Completed split 50/50
Finished in 67.1 minutes
Performance rows: 300; fidelity rows: 450


## 6. Metrics and corrected resampled confidence intervals

ROC-AUC is primary. Average precision is the PR summary, and sensitivity is calculated at each estimator's default classification threshold. Pairwise ROC-AUC differences use the Nadeau–Bengio corrected resampled t interval. The same correction is applied to fidelity means; error-agreement intervals use only folds with at least one black-box error and report that effective fold count.

In [12]:
from core import summarize_results
from IPython.display import display

summaries = summarize_results(model_results, fidelity_results,
                              n_splits=N_SPLITS, n_repeats=N_REPEATS)
performance_summary = summaries["performance_summary"]
auc_comparisons = summaries["auc_comparisons"]
fidelity_summary = summaries["fidelity_summary"]
display(performance_summary.round(4))
display(auc_comparisons.round(4))
display(fidelity_summary.round(4))

,roc_auc_mean,roc_auc_sd,average_precision_mean,sensitivity_mean,n_folds
model,,,,,
Depth-limited CART,0.8094,0.0437,0.1779,0.7758,50
Explainable Boosting Machine,0.8431,0.0306,0.2152,0.8159,50
Gradient boosting,0.8499,0.0283,0.2406,0.8199,50
L1 logistic regression,0.8482,0.0297,0.2438,0.7983,50
Multilayer perceptron,0.8416,0.0296,0.2227,0.7968,50
Random forest,0.8259,0.0294,0.2078,0.0827,50


,interpretable_model,black_box_model,mean_difference,ci_low,ci_high,n_folds
0,L1 logistic regression,Random forest,0.0222,0.0093,0.0352,50
1,L1 logistic regression,Gradient boosting,-0.0018,-0.0110,0.0075,50
2,L1 logistic regression,Multilayer perceptron,0.0066,-0.0024,0.0155,50
3,Explainable Boosting Machine,Random forest,0.0172,0.0027,0.0318,50
4,Explainable Boosting Machine,Gradient boosting,-0.0068,-0.0160,0.0024,50
5,Explainable Boosting Machine,Multilayer perceptron,0.0016,-0.0094,0.0125,50
6,Depth-limited CART,Random forest,-0.0165,-0.0429,0.0099,50
7,Depth-limited CART,Gradient boosting,-0.0406,-0.0636,-0.0176,50
8,Depth-limited CART,Multilayer perceptron,-0.0322,-0.0592,-0.0051,50


,black_box,surrogate,agreement_mean,agreement_ci_low,agreement_ci_high,agreement_n_folds,error_agreement_mean,error_agreement_ci_low,error_agreement_ci_high,error_agreement_n_folds,majority_baseline_mean,mean_black_box_errors
0,Gradient boosting,Depth-limited CART,0.9728,0.9634,0.9822,50,0.9737,0.9618,0.9855,50,0.6916,141.64
1,Gradient boosting,Explainable Boosting Machine,0.9912,0.9872,0.9951,50,0.9885,0.9813,0.9956,50,0.6916,141.64
2,Gradient boosting,L1 logistic regression,0.9561,0.9463,0.9659,50,0.9614,0.9463,0.9765,50,0.6916,141.64
3,Multilayer perceptron,Depth-limited CART,0.9324,0.9179,0.9469,50,0.9357,0.9140,0.9574,50,0.7130,131.90
4,Multilayer perceptron,Explainable Boosting Machine,0.9683,0.9564,0.9802,50,0.9645,0.9481,0.9808,50,0.7130,131.90
5,Multilayer perceptron,L1 logistic regression,0.9642,0.9522,0.9762,50,0.9650,0.9491,0.9808,50,0.7130,131.90
6,Random forest,Depth-limited CART,0.7239,0.6992,0.7485,50,0.3691,0.2959,0.4424,50,0.9855,28.18
7,Random forest,Explainable Boosting Machine,0.7289,0.7142,0.7436,50,0.3594,0.2907,0.4282,50,0.9855,28.18
8,Random forest,L1 logistic regression,0.7432,0.7297,0.7568,50,0.3678,0.3041,0.4316,50,0.9855,28.18


## 7. Inspect fold-level fidelity and error agreement

Overall agreement measures how often the surrogate matches the black-box on held-out rows. Error agreement conditions on black-box mistakes: it is the proportion of those mistakes the surrogate repeats. It is undefined when that fold contains no black-box errors. Majority-baseline agreement provides a simple reference.

In [13]:
display(fidelity_results.head(12))
display(fidelity_results.groupby(["black_box", "surrogate"])[
    ["agreement", "error_agreement", "majority_baseline_agreement", "black_box_error_count"]
].agg(["mean", "std", "count"]).round(4))

,split,black_box,surrogate,agreement,error_agreement,majority_baseline_agreement,black_box_error_count
0,1,Random forest,L1 logistic regression,0.761252,0.354839,0.984344,31
1,1,Random forest,Depth-limited CART,0.698630,0.290323,0.984344,31
2,1,Random forest,Explainable Boosting Machine,0.739726,0.322581,0.984344,31
3,1,Gradient boosting,L1 logistic regression,0.951076,0.960000,0.714286,125
4,1,Gradient boosting,Depth-limited CART,0.956947,0.968000,0.714286,125
5,1,Gradient boosting,Explainable Boosting Machine,0.978474,0.960000,0.714286,125
6,1,Multilayer perceptron,L1 logistic regression,0.964775,0.964286,0.743640,112
7,1,Multilayer perceptron,Depth-limited CART,0.923679,0.928571,0.743640,112
8,1,Multilayer perceptron,Explainable Boosting Machine,0.980431,0.955357,0.743640,112
9,2,Random forest,L1 logistic regression,0.765166,0.500000,0.978474,28


agreement                \
                                                        mean     std count   
black_box             surrogate                                              
Gradient boosting     Depth-limited CART              0.9728  0.0129    50   
                      Explainable Boosting Machine    0.9912  0.0054    50   
                      L1 logistic regression          0.9561  0.0134    50   
Multilayer perceptron Depth-limited CART              0.9324  0.0199    50   
                      Explainable Boosting Machine    0.9683  0.0164    50   
                      L1 logistic regression          0.9642  0.0165    50   
Random forest         Depth-limited CART              0.7239  0.0338    50   
                      Explainable Boosting Machine    0.7289  0.0202    50   
                      L1 logistic regression          0.7432  0.0186    50   

                                                   error_agreement          \
                                                              mean     std   
black_box             surrogate                                              
Gradient boosting     Depth-limited CART                    0.9737  0.0163   
                      Explainable Boosting Machine          0.9885  0.0098   
                      L1 logistic regression                0.9614  0.0207   
Multilayer perceptron Depth-limited CART                    0.9357  0.0299   
                      Explainable Boosting Machine          0.9645  0.0225   
                      L1 logistic regression                0.9650  0.0218   
Random forest         Depth-limited CART                    0.3691  0.1006   
                      Explainable Boosting Machine          0.3594  0.0945   
                      L1 logistic regression                0.3678  0.0876   

                                                          \
                                                   count   
black_box             surrogate                            
Gradient boosting     Depth-limited CART              50   
                      Explainable Boosting Machine    50   
                      L1 logistic regression          50   
Multilayer perceptron Depth-limited CART              50   
                      Explainable Boosting Machine    50   
                      L1 logistic regression          50   
Random forest         Depth-limited CART              50   
                      Explainable Boosting Machine    50   
                      L1 logistic regression          50   

                                                   majority_baseline_agreement  \
                                                                          mean   
black_box             surrogate                                                  
Gradient boosting     Depth-limited CART                                0.6916   
                      Explainable Boosting Machine                      0.6916   
                      L1 logistic regression                            0.6916   
Multilayer perceptron Depth-limited CART                                0.7130   
                      Explainable Boosting Machine                      0.7130   
                      L1 logistic regression                            0.7130   
Random forest         Depth-limited CART                                0.9855   
                      Explainable Boosting Machine                      0.9855   
                      L1 logistic regression                            0.9855   

                                                                  \
                                                       std count   
black_box             surrogate                                    
Gradient boosting     Depth-limited CART            0.0183    50   
                      Explainable Boosting Machine  0.0183    50   
                      L1 logistic regression        0.0183    50   
Multilayer perceptron Depth-limited CART            0.0377    5

## 8. Save and download CSV results

The five CSVs include fold-level model performance, fold-level fidelity, performance summary, pairwise AUC comparisons, and fidelity summaries. They are saved under `/content/Downloads/stroke_model_comparison_results/`; a ZIP is also saved in `/content/Downloads/` and downloaded by the browser to your computer's default Downloads folder.

In [14]:
from pathlib import Path
import zipfile
from google.colab import files

DOWNLOADS_DIR = Path("/content/Downloads")
RESULTS_DIR = DOWNLOADS_DIR / "stroke_model_comparison_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
model_results.to_csv(RESULTS_DIR / "model_fold_results.csv", index=False)
fidelity_results.to_csv(RESULTS_DIR / "fidelity_fold_results.csv", index=False)
performance_summary.to_csv(RESULTS_DIR / "performance_summary.csv")
auc_comparisons.to_csv(RESULTS_DIR / "auc_comparisons.csv", index=False)
fidelity_summary.to_csv(RESULTS_DIR / "fidelity_summary.csv", index=False)
zip_path = DOWNLOADS_DIR / "stroke_model_comparison_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in RESULTS_DIR.glob("*.csv"):
        archive.write(path, arcname=path.name)
print("Saved files:", [p.name for p in sorted(RESULTS_DIR.glob("*.csv"))])
files.download(str(zip_path))

Saved files: ['auc_comparisons.csv', 'fidelity_fold_results.csv', 'fidelity_summary.csv', 'model_fold_results.csv', 'performance_summary.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reproducibility notes

- Fixed random seed: 42; the CV splitter and stochastic estimators use deterministic seeds.
- Imputation, scaling, one-hot encoding, and GOSDT threshold learning are fitted within training folds.
- A missing error-agreement estimate means that the black-box made no held-out mistakes in that fold.
- Optional EBM/GOSDT omissions are printed and appear in the available-model listing. Do not describe an omitted model as evaluated.
- GOSDT uses hard predictions for ROC-AUC and average precision here; those values are coarser than probability-based scores.
- The corrected intervals are approximate and account for overlap between resampled training sets; they are not clinical uncertainty intervals.
- Record the Colab runtime, package versions, `QUICK_RUN`, and optional model availability when reporting results.